In [1]:
# Load the five datasets into an in-memory SQLite database
from pathlib import Path
import pandas as pd
import sqlite3

data_dir = Path("data")
conn = sqlite3.connect(":memory:")

tables = [
    "users", "listening_events", "subscription_events",
    "ad_events", "feature_table"
]

for table in tables:
    df = pd.read_csv(data_dir / f"{table}.csv")
    df.to_sql(table, conn, if_exists="replace", index=False)

def sql(query):
    return pd.read_sql_query(query, conn)

def run_script(query):
    conn.executescript(query)
    print("SQL script executed successfully.")

print("Database ready!")

Database ready!


## Section 1 | Time Window Design

Define the snapshot date, observation window, and prediction windows.

In [2]:
# Define observation and prediction windows to prevent data leakage
query = """
SELECT
    '2026-04-01' AS snapshot_date,
    '2026-03-02' AS observation_start,
    '2026-04-01' AS observation_end_exclusive,
    '2026-04-15' AS churn_prediction_end_exclusive,
    '2026-05-01' AS conversion_prediction_end_exclusive
;"""

sql(query)

,snapshot_date,observation_start,observation_end_exclusive,churn_prediction_end_exclusive,conversion_prediction_end_exclusive
0,2026-04-01,2026-03-02,2026-04-01,2026-04-15,2026-05-01


## Section 2 | Base Users

Construct a user-level dataset using the latest subscription status before the snapshot date.

In [3]:
# Retrieve each user's latest subscription status before the snapshot
query = """
WITH latest_subscription AS (
    SELECT
        user_id,
        plan_type AS latest_plan_type,
        event_type AS latest_subscription_event
    FROM (
        SELECT
            user_id,
            plan_type,
            event_type,
            event_timestamp,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY event_timestamp DESC
            ) AS rn
        FROM subscription_events
        WHERE event_timestamp < '2026-04-01'
    )
    WHERE rn = 1
)
SELECT
    u.user_id,
    u.signup_date,
    u.country,
    u.primary_device,
    u.acquisition_channel,
    COALESCE(l.latest_plan_type, 'free') AS subscription_type,
    '2026-04-01' AS snapshot_date
FROM users u
LEFT JOIN latest_subscription l
    ON u.user_id = l.user_id
LIMIT 10
;"""

sql(query)

,user_id,signup_date,country,primary_device,acquisition_channel,subscription_type,snapshot_date
0,U000001,2026-02-06,UK,mobile_android,organic_search,individual_premium,2026-04-01
1,U000002,2026-03-07,ES,mobile_android,paid_social,trial,2026-04-01
2,U000003,2025-07-29,IN,mobile_android,referral,trial,2026-04-01
3,U000004,2025-11-14,FR,mobile_android,referral,trial,2026-04-01
4,U000005,2025-05-21,US,mobile_ios,student_partner,free,2026-04-01
5,U000006,2026-03-09,US,mobile_android,email_campaign,trial,2026-04-01
6,U000007,2025-12-17,US,desktop,email_campaign,trial,2026-04-01
7,U000008,2026-01-19,MX,desktop,app_store,free,2026-04-01
8,U000009,2026-03-06,FR,mobile_ios,paid_social,trial,2026-04-01
9,U000010,2025-12-21,IN,mobile_android,paid_social,individual_premium,2026-04-01


## Section 3 | Listening Features

Aggregate listening events into user-level behavioral features over the 30-day observation window.

In [5]:
# Aggregate listening behavior within the 30-day observation window
query = """
SELECT
    user_id,
    COUNT(DISTINCT DATE(event_timestamp)) AS active_days_30d,
    COUNT(*) AS listen_events_30d,
    SUM(COALESCE(play_duration_sec, 0)) / 60.0 AS listen_minutes_30d,
    COUNT(DISTINCT session_id) AS sessions_30d,
    AVG(CASE WHEN skipped_flag = 1 THEN 1.0 ELSE 0.0 END) AS skip_rate_30d,
    AVG(CASE WHEN completed_flag = 1 THEN 1.0 ELSE 0.0 END) AS completion_rate_30d,
    SUM(CASE WHEN liked_flag = 1 THEN 1 ELSE 0 END) AS liked_songs_30d,
    SUM(CASE WHEN playlist_add_flag = 1 THEN 1 ELSE 0 END) AS playlist_adds_30d,
    SUM(CASE WHEN search_used_flag = 1 THEN 1 ELSE 0 END) AS search_events_30d
FROM listening_events
WHERE event_timestamp >= '2026-03-02'
  AND event_timestamp < '2026-04-01'
GROUP BY user_id
LIMIT 10
;"""

sql(query)

,user_id,active_days_30d,listen_events_30d,listen_minutes_30d,sessions_30d,skip_rate_30d,completion_rate_30d,liked_songs_30d,playlist_adds_30d,search_events_30d
0,U000001,1,1,1.933333,1,0.000000,0.000000,0,0,0
1,U000002,10,10,21.550000,10,0.200000,0.300000,1,1,2
2,U000003,3,3,6.500000,3,0.000000,0.000000,0,0,0
3,U000004,4,4,11.333333,4,0.250000,0.000000,0,0,3
4,U000005,1,1,2.716667,1,0.000000,0.000000,0,0,0
5,U000006,3,3,32.633333,3,0.000000,0.666667,0,0,1
6,U000007,20,34,226.950000,31,0.117647,0.382353,1,1,6
7,U000008,3,3,33.416667,3,0.000000,0.000000,0,0,1
8,U000009,13,17,101.916667,15,0.176471,0.470588,2,2,5
9,U000010,3,3,3.416667,3,0.666667,0.333333,0,0,1


## Section 4 | Ad Features

Aggregate ad impressions, clicks, completions, and revenue into user-level features.

In [6]:
# Aggregate ad exposure and engagement over the observation window
query = """
SELECT
    user_id,
    COUNT(*) AS ad_impressions_30d,
    SUM(CASE WHEN clicked_flag = 1 THEN 1 ELSE 0 END) AS ad_clicks_30d,
    1.0 * SUM(CASE WHEN clicked_flag = 1 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0) AS ad_click_rate_30d,
    SUM(CASE WHEN completed_flag = 1 THEN 1 ELSE 0 END) AS ad_completions_30d,
    1.0 * SUM(CASE WHEN completed_flag = 1 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0) AS ad_completion_rate_30d,
    SUM(COALESCE(revenue_usd, 0)) AS ad_revenue_30d
FROM ad_events
WHERE event_timestamp >= '2026-03-02'
  AND event_timestamp < '2026-04-01'
GROUP BY user_id
LIMIT 10
;"""

sql(query)

,user_id,ad_impressions_30d,ad_clicks_30d,ad_click_rate_30d,ad_completions_30d,ad_completion_rate_30d,ad_revenue_30d
0,U000001,3,0,0.0,2,0.666667,0.0438
1,U000002,7,0,0.0,6,0.857143,0.0544
2,U000003,1,0,0.0,0,0.000000,0.0025
3,U000004,2,0,0.0,1,0.500000,0.0703
4,U000006,4,0,0.0,2,0.500000,0.0671
5,U000007,16,0,0.0,9,0.562500,0.2380
6,U000008,2,0,0.0,1,0.500000,0.0106
7,U000009,4,2,0.5,3,0.750000,0.1573
8,U000011,1,0,0.0,1,1.000000,0.0096
9,U000012,2,0,0.0,2,1.000000,0.0256


## Section 5 | Subscription Features

Aggregate trial activity, paid subscriptions, renewals, payment failures, and cancellations over the 30-day observation window.

In [7]:
# Aggregate subscription events into user-level features
query = """
SELECT
    user_id,
    MAX(CASE WHEN event_type = 'trial_exposed' THEN 1 ELSE 0 END) AS trial_exposed_30d,
    MAX(CASE WHEN event_type = 'trial_started' THEN 1 ELSE 0 END) AS trial_started_30d,
    MAX(CASE WHEN event_type = 'paid_started' THEN 1 ELSE 0 END) AS paid_started_in_observation_30d,
    SUM(CASE WHEN event_type = 'renewal_success' THEN 1 ELSE 0 END) AS renewal_success_count_30d,
    SUM(CASE WHEN event_type = 'payment_failed' THEN 1 ELSE 0 END) AS payment_failed_count_30d,
    SUM(CASE WHEN event_type = 'cancel' THEN 1 ELSE 0 END) AS cancel_count_30d,
    -- Estimate revenue using successful payment events only
    SUM(CASE
        WHEN event_type IN ('paid_started', 'renewal_success')
        THEN COALESCE(price_usd, 0)
        ELSE 0
    END) AS subscription_revenue_observation_30d
FROM subscription_events
WHERE event_timestamp >= '2026-03-02'
  AND event_timestamp < '2026-04-01'
GROUP BY user_id
LIMIT 10
;"""

sql(query)

,user_id,trial_exposed_30d,trial_started_30d,paid_started_in_observation_30d,renewal_success_count_30d,payment_failed_count_30d,cancel_count_30d,subscription_revenue_observation_30d
0,U000001,0,0,0,0,0,1,0.00
1,U000002,1,0,0,0,0,0,0.00
2,U000006,1,0,0,0,0,0,0.00
3,U000007,1,0,0,0,0,0,0.00
4,U000009,1,0,1,0,0,0,10.99
5,U000010,0,0,0,1,0,0,10.99
6,U000015,1,1,0,0,0,0,0.00
7,U000016,1,0,0,1,0,0,10.99
8,U000017,1,0,1,0,0,0,10.99
9,U000020,0,0,0,1,0,0,16.99


## Section 6 | Churn & Conversion Labels

Define churn and paid conversion outcomes using future prediction windows.

In [8]:
# Define churn and conversion labels using future user activity
query = """
WITH churn_label AS (
    SELECT
        u.user_id,
        CASE
            WHEN COUNT(l.user_id) = 0 THEN 1
            ELSE 0
        END AS churn_label_14d
    FROM users u
    LEFT JOIN listening_events l
        ON u.user_id = l.user_id
       AND l.event_timestamp >= '2026-04-01'
       AND l.event_timestamp < '2026-04-15'
    GROUP BY u.user_id
),
conversion_label AS (
    SELECT
        u.user_id,
        CASE
            WHEN COUNT(s.user_id) > 0 THEN 1
            ELSE 0
        END AS paid_conversion_30d
    FROM users u
    LEFT JOIN subscription_events s
        ON u.user_id = s.user_id
       AND s.event_timestamp >= '2026-04-01'
       AND s.event_timestamp < '2026-05-01'
       AND s.event_type = 'paid_started'
    GROUP BY u.user_id
)
SELECT
    c.user_id,
    c.churn_label_14d,
    p.paid_conversion_30d
FROM churn_label c
LEFT JOIN conversion_label p
    ON c.user_id = p.user_id
LIMIT 10
;"""

sql(query)

,user_id,churn_label_14d,paid_conversion_30d
0,U000001,0,0
1,U000002,1,0
2,U000003,1,0
3,U000004,0,0
4,U000005,0,0
5,U000006,0,0
6,U000007,0,0
7,U000008,1,0
8,U000009,0,0
9,U000010,0,0


## Section 7 | Final User-Level Feature Table

Combine user attributes, listening behavior, ad engagement,
subscription activity, and prediction labels into a user-level dataset.

In [9]:
# Build the final user-level feature table
query = """
DROP VIEW IF EXISTS user_level_feature_table;

CREATE TEMP VIEW user_level_feature_table AS

WITH latest_subscription AS (
    SELECT
        user_id,
        plan_type AS latest_plan_type,
        event_type AS latest_subscription_event
    FROM (
        SELECT
            user_id,
            plan_type,
            event_type,
            event_timestamp,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY event_timestamp DESC
            ) AS rn
        FROM subscription_events
        WHERE event_timestamp < '2026-04-01'
    )
    WHERE rn = 1
),

base_users AS (
    SELECT
        u.user_id,
        '2026-04-01' AS snapshot_date,
        u.signup_date,
        CAST(julianday('2026-04-01') - julianday(u.signup_date) AS INTEGER) AS tenure_days,
        u.country,
        u.primary_device,
        u.acquisition_channel,
        COALESCE(l.latest_plan_type, 'free') AS subscription_type
    FROM users u
    LEFT JOIN latest_subscription l
        ON u.user_id = l.user_id
),

listening_features AS (
    SELECT
        user_id,
        COUNT(DISTINCT DATE(event_timestamp)) AS active_days_30d,
        COUNT(*) AS listen_events_30d,
        SUM(COALESCE(play_duration_sec, 0)) / 60.0 AS listen_minutes_30d,
        COUNT(DISTINCT session_id) AS sessions_30d,
        AVG(COALESCE(play_duration_sec, 0)) AS avg_play_duration_sec_30d,
        AVG(CASE WHEN skipped_flag = 1 THEN 1.0 ELSE 0.0 END) AS skip_rate_30d,
        AVG(CASE WHEN completed_flag = 1 THEN 1.0 ELSE 0.0 END) AS completion_rate_30d,
        SUM(CASE WHEN liked_flag = 1 THEN 1 ELSE 0 END) AS liked_songs_30d,
        SUM(CASE WHEN playlist_add_flag = 1 THEN 1 ELSE 0 END) AS playlist_adds_30d,
        SUM(CASE WHEN search_used_flag = 1 THEN 1 ELSE 0 END) AS search_events_30d
    FROM listening_events
    WHERE event_timestamp >= '2026-03-02'
      AND event_timestamp < '2026-04-01'
    GROUP BY user_id
),

genre_counts AS (
    SELECT
        user_id,
        genre,
        COUNT(*) AS genre_events
    FROM listening_events
    WHERE event_timestamp >= '2026-03-02'
      AND event_timestamp < '2026-04-01'
      AND genre IS NOT NULL
    GROUP BY user_id, genre
),

top_genre AS (
    SELECT
        user_id,
        genre AS top_genre_30d
    FROM (
        SELECT
            user_id,
            genre,
            genre_events,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY genre_events DESC, genre
            ) AS rn
        FROM genre_counts
    )
    WHERE rn = 1
),

ad_features AS (
    SELECT
        user_id,
        COUNT(*) AS ad_impressions_30d,
        SUM(CASE WHEN clicked_flag = 1 THEN 1 ELSE 0 END) AS ad_clicks_30d,
        1.0 * SUM(CASE WHEN clicked_flag = 1 THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0) AS ad_click_rate_30d,
        SUM(CASE WHEN completed_flag = 1 THEN 1 ELSE 0 END) AS ad_completions_30d,
        1.0 * SUM(CASE WHEN completed_flag = 1 THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0) AS ad_completion_rate_30d,
        SUM(COALESCE(revenue_usd, 0)) AS ad_revenue_30d
    FROM ad_events
    WHERE event_timestamp >= '2026-03-02'
      AND event_timestamp < '2026-04-01'
    GROUP BY user_id
),

subscription_features AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_type = 'trial_exposed' THEN 1 ELSE 0 END) AS trial_exposed_30d,
        MAX(CASE WHEN event_type = 'trial_started' THEN 1 ELSE 0 END) AS trial_started_30d,
        MAX(CASE WHEN event_type = 'paid_started' THEN 1 ELSE 0 END) AS paid_started_30d,
        SUM(CASE WHEN event_type = 'renewal_success' THEN 1 ELSE 0 END) AS renewal_success_30d,
        SUM(CASE WHEN event_type = 'payment_failed' THEN 1 ELSE 0 END) AS payment_failed_30d,
        SUM(CASE WHEN event_type = 'cancel' THEN 1 ELSE 0 END) AS cancel_30d,
        SUM(CASE WHEN event_type = 'trial_expired' THEN 1 ELSE 0 END) AS trial_expired_30d,
        -- Estimate revenue from successful payment events
        SUM(CASE
            WHEN event_type IN ('paid_started', 'renewal_success')
            THEN COALESCE(price_usd, 0)
            ELSE 0
        END) AS subscription_revenue_30d
    FROM subscription_events
    WHERE event_timestamp >= '2026-03-02'
      AND event_timestamp < '2026-04-01'
    GROUP BY user_id
),

churn_label AS (
    SELECT
        u.user_id,
        CASE WHEN COUNT(l.user_id) = 0 THEN 1 ELSE 0 END AS churn_label_14d
    FROM users u
    LEFT JOIN listening_events l
        ON u.user_id = l.user_id
       AND l.event_timestamp >= '2026-04-01'
       AND l.event_timestamp < '2026-04-15'
    GROUP BY u.user_id
),

conversion_label AS (
    SELECT
        u.user_id,
        CASE WHEN COUNT(s.user_id) > 0 THEN 1 ELSE 0 END AS paid_conversion_30d
    FROM users u
    LEFT JOIN subscription_events s
        ON u.user_id = s.user_id
       AND s.event_timestamp >= '2026-04-01'
       AND s.event_timestamp < '2026-05-01'
       AND s.event_type = 'paid_started'
    GROUP BY u.user_id
)

SELECT
    b.user_id,
    b.snapshot_date,
    b.signup_date,
    b.tenure_days,
    b.country,
    b.primary_device,
    b.acquisition_channel,
    b.subscription_type,

    -- Listening features
    COALESCE(l.active_days_30d, 0) AS active_days_30d,
    COALESCE(l.listen_events_30d, 0) AS listen_events_30d,
    COALESCE(l.listen_minutes_30d, 0) AS listen_minutes_30d,
    COALESCE(l.sessions_30d, 0) AS sessions_30d,
    COALESCE(l.avg_play_duration_sec_30d, 0) AS avg_play_duration_sec_30d,
    COALESCE(l.skip_rate_30d, 0) AS skip_rate_30d,
    COALESCE(l.completion_rate_30d, 0) AS completion_rate_30d,
    COALESCE(l.liked_songs_30d, 0) AS liked_songs_30d,
    COALESCE(l.playlist_adds_30d, 0) AS playlist_adds_30d,
    COALESCE(l.search_events_30d, 0) AS search_events_30d,
    tg.top_genre_30d,

    -- Ad features
    COALESCE(a.ad_impressions_30d, 0) AS ad_impressions_30d,
    COALESCE(a.ad_clicks_30d, 0) AS ad_clicks_30d,
    COALESCE(a.ad_click_rate_30d, 0) AS ad_click_rate_30d,
    COALESCE(a.ad_completions_30d, 0) AS ad_completions_30d,
    COALESCE(a.ad_completion_rate_30d, 0) AS ad_completion_rate_30d,
    CASE
        WHEN COALESCE(l.active_days_30d, 0) > 0
        THEN 1.0 * COALESCE(a.ad_impressions_30d, 0) / l.active_days_30d
        ELSE 0
    END AS ad_impressions_per_active_day,
    COALESCE(a.ad_revenue_30d, 0) AS ad_revenue_30d,

    -- Subscription features
    COALESCE(s.trial_exposed_30d, 0) AS trial_exposed_30d,
    COALESCE(s.trial_started_30d, 0) AS trial_started_30d,
    COALESCE(s.paid_started_30d, 0) AS paid_started_30d,
    COALESCE(s.renewal_success_30d, 0) AS renewal_success_30d,
    COALESCE(s.payment_failed_30d, 0) AS payment_failed_30d,
    COALESCE(s.cancel_30d, 0) AS cancel_30d,
    COALESCE(s.trial_expired_30d, 0) AS trial_expired_30d,
    COALESCE(s.subscription_revenue_30d, 0) AS subscription_revenue_30d,
    COALESCE(a.ad_revenue_30d, 0)
        + COALESCE(s.subscription_revenue_30d, 0) AS total_revenue_30d,

    -- Future prediction labels
    c.churn_label_14d,
    p.paid_conversion_30d

FROM base_users b
LEFT JOIN listening_features l ON b.user_id = l.user_id
LEFT JOIN top_genre tg ON b.user_id = tg.user_id
LEFT JOIN ad_features a ON b.user_id = a.user_id
LEFT JOIN subscription_features s ON b.user_id = s.user_id
LEFT JOIN churn_label c ON b.user_id = c.user_id
LEFT JOIN conversion_label p ON b.user_id = p.user_id
;"""

run_script(query)

# Preview the final wide table
sql("SELECT * FROM user_level_feature_table LIMIT 5")

SQL script executed successfully.


,user_id,snapshot_date,signup_date,tenure_days,country,primary_device,acquisition_channel,subscription_type,active_days_30d,listen_events_30d,...,trial_started_30d,paid_started_30d,renewal_success_30d,payment_failed_30d,cancel_30d,trial_expired_30d,subscription_revenue_30d,total_revenue_30d,churn_label_14d,paid_conversion_30d
0,U000001,2026-04-01,2026-02-06,54,UK,mobile_android,organic_search,individual_premium,1,1,...,0,0,0,0,1,0,0,0.0438,0,0
1,U000002,2026-04-01,2026-03-07,25,ES,mobile_android,paid_social,trial,10,10,...,0,0,0,0,0,0,0,0.0544,1,0
2,U000003,2026-04-01,2025-07-29,246,IN,mobile_android,referral,trial,3,3,...,0,0,0,0,0,0,0,0.0025,1,0
3,U000004,2026-04-01,2025-11-14,138,FR,mobile_android,referral,trial,4,4,...,0,0,0,0,0,0,0,0.0703,0,0
4,U000005,2026-04-01,2025-05-21,315,US,mobile_ios,student_partner,free,1,1,...,0,0,0,0,0,0,0,0.0000,0,0


## Section 8 | Data Quality Check: Duplicate Users

Verify that the final feature table maintains one row per user.

In [10]:
# Verify that JOIN operations did not duplicate user records
query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS user_count,
    COUNT(*) - COUNT(DISTINCT user_id) AS duplicated_rows
FROM user_level_feature_table
;"""

sql(query)

,row_count,user_count,duplicated_rows
0,50000,50000,0


## Section 9 | Data Quality Check: Feature & Label Distribution

Validate the distribution of key features and prediction labels in the final user-level dataset.

In [11]:
# Validate churn, conversion, and engagement metrics
query = """
SELECT
    AVG(churn_label_14d) AS churn_rate_14d,
    AVG(paid_conversion_30d) AS paid_conversion_rate_30d,
    AVG(active_days_30d) AS avg_active_days_30d,
    AVG(listen_minutes_30d) AS avg_listen_minutes_30d,
    AVG(ad_impressions_per_active_day) AS avg_ads_per_active_day
FROM user_level_feature_table
;"""

sql(query)

,churn_rate_14d,paid_conversion_rate_30d,avg_active_days_30d,avg_listen_minutes_30d,avg_ads_per_active_day
0,0.47558,0.0,6.23184,37.316219,0.488783


**QA Note:** The SQL-built paid conversion label contains no positive
cases because no `paid_started` events occur within the future
30-day prediction window. This differs from the precomputed
feature table and should be treated as a label-definition limitation.

## Section 10 | Export User-Level Feature Table

Export the SQL-built dataset for EDA and churn prediction.

In [12]:
# Export the final user-level dataset for EDA and modeling
wide_df = sql("SELECT * FROM user_level_feature_table")

output_path = data_dir / "user_level_feature_table_sql_built.csv"
wide_df.to_csv(output_path, index=False)

print(f"Exported to: {output_path}")
print(f"Dataset shape: {wide_df.shape}")

wide_df.head()

Exported to: data/user_level_feature_table_sql_built.csv
Dataset shape: (50000, 37)


,user_id,snapshot_date,signup_date,tenure_days,country,primary_device,acquisition_channel,subscription_type,active_days_30d,listen_events_30d,...,trial_started_30d,paid_started_30d,renewal_success_30d,payment_failed_30d,cancel_30d,trial_expired_30d,subscription_revenue_30d,total_revenue_30d,churn_label_14d,paid_conversion_30d
0,U000001,2026-04-01,2026-02-06,54,UK,mobile_android,organic_search,individual_premium,1,1,...,0,0,0,0,1,0,0.0,0.0438,0,0
1,U000002,2026-04-01,2026-03-07,25,ES,mobile_android,paid_social,trial,10,10,...,0,0,0,0,0,0,0.0,0.0544,1,0
2,U000003,2026-04-01,2025-07-29,246,IN,mobile_android,referral,trial,3,3,...,0,0,0,0,0,0,0.0,0.0025,1,0
3,U000004,2026-04-01,2025-11-14,138,FR,mobile_android,referral,trial,4,4,...,0,0,0,0,0,0,0.0,0.0703,0,0
4,U000005,2026-04-01,2025-05-21,315,US,mobile_ios,student_partner,free,1,1,...,0,0,0,0,0,0,0.0,0.0000,0,0
